In [1]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.EmpennageReq import EmpennageReq
from Requirements.Requirement import Requirement
from EmpennageSizing.TailFinder import TailFinder
from EmpennageSizing.CanardFinder import CanardFinder
from structural_analysis.Material import Material

# Loading the pre-computed planforms and the fuselage

In [2]:
with open("pickles/planform_pickle_official.pcl", "r+b") as f:
    plaforms_recovered:list[tuple[Planform, str, bool]] = pickle.load(f)

assumptions = Assumptions()

In [3]:
# # --- Import Onshape pull utilities ---
# sys.path.append(os.path.abspath(os.getcwd()))
# from onshape_pull import (
#     fetch_variable_studio, fetch_measurement_features,
#     evaluate_measurements, load_cached_masses, fetch_mass_properties,
#     compute_cg_scenarios, lookup_var, lookup_meas,
#     UPDATE_MASSES,
# )

# # --- Pull data from Onshape ---
# variables = fetch_variable_studio()
# meas_names = fetch_measurement_features()
# measurements = evaluate_measurements(meas_names)
# components = load_cached_masses() if not UPDATE_MASSES else fetch_mass_properties()
# cg_data = compute_cg_scenarios(components)

# # --- Z offset (axle datum) ---
# Front_Landing_Gear_Hinge_Z = lookup_meas(measurements, "Front_Landing_Gear_Hinge_Z")
# Front_Strut_Height, _, _ = lookup_var(variables, "Front_Strut_Height")
# Front_Gear_Extension_Max = lookup_meas(measurements, "Front_Gear_Extension_Max")
# Front_Gear_Extension_Min = lookup_meas(measurements, "Front_Gear_Extension_Min")
# z_offset = (Front_Landing_Gear_Hinge_Z + Front_Strut_Height
#             + (Front_Gear_Extension_Max - Front_Gear_Extension_Min))

# # --- Build drag components ---
# engine_bay = Bay(
#     surface_wetted=83744.32631 / 1e6,  # mm² → m² (hardcoded, not in Onshape)
#     length=0.172,                       # 172 mm (hardcoded, not in Onshape)
#     diameter=lookup_var(variables, "engine_diameter")[0],
# )

# Front_Gear_Unexposed = lookup_meas(measurements, "Front_Gear_Unexposed")
# nose_gear = LandingGear(
#     wheel_width=0.025,
#     exposed_height=Front_Strut_Height - Front_Gear_Unexposed,
#     wheel_diameter=lookup_var(variables, "Wheel_Diameter")[0],
#     strut_width=lookup_var(variables, "Front_Strut_Diameter")[0],
# )

# Rear_Strut_Height, _, _ = lookup_var(variables, "Rear_Strut_Height")
# Rear_Strut_height_2, _, _ = lookup_var(variables, "Rear_Strut_height_2")
# main_gear = LandingGear(
#     wheel_width=0.025,
#     exposed_height=Rear_Strut_Height + Rear_Strut_height_2,
#     wheel_diameter=lookup_var(variables, "Wheel_Diameter")[0],
#     strut_width=lookup_var(variables, "Rear_Strut_Diameter")[0],
# )

# fuselage = Fuselage(
#     surface_wetted=lookup_meas(measurements, "Wetted_Area"),
#     length_total=lookup_var(variables, "FuselageLength")[0],
#     diameter_max=lookup_var(variables, "FuselageHeight")[0],
#     upsweep=0.0,
#     base_area=lookup_meas(measurements, "Base_Area"),
# )

# # --- X-position helpers ---
# WingPortDistance, _, _ = lookup_var(variables, "FuselageLength")
# WingPortDistance = (WingPortDistance / 2) - 0.025
# WingPortWidth, _, _ = lookup_var(variables, "WingPortWidth")
# CanardPortXLoc, _, _ = lookup_var(variables, "CanardPortXLoc")
# CanardPortWidth, _, _ = lookup_var(variables, "CanardPortWidth")

# print(cg_data["x_cg_min"])
# print(cg_data["x_cg_max"])


# # --- Fixed parameters ---
# fixed = Fixed(
#     mass=cg_data["mass"],
#     fuel_mass=cg_data["fuel_mass"],
#     x_cg_min=cg_data["x_cg_min"],
#     x_cg_max=cg_data["x_cg_max"],
#     x_tail_cone=lookup_meas(measurements, "Tailcone_X"),
#     z_cg=cg_data["z_cg_full"] + z_offset,
#     z_tail_cone=-lookup_meas(measurements, "Z_TailCone") + z_offset,
#     z_wing=lookup_meas(measurements, "Z_wing_LE_Abs") + z_offset,
#     x_LE_canard=CanardPortXLoc + CanardPortWidth / 2,
#     x_LE_wing=WingPortDistance + 0.115 - int(WingPortWidth) / 2,
#     x_LE_tail=lookup_meas(measurements, "X_LE_Tail"),
#     x_nose_gear=lookup_meas(measurements, "x_nose_gear"),
#     x_main_gear=lookup_meas(measurements, "x_main_gear"),
#     y_main_gear=0.419,
#     fuselage=fuselage,
#     nose_gear=nose_gear,
#     main_gear=main_gear,
#     engine_bay=engine_bay,
# )

In [4]:
# with open("pickles/fixed_pickle.pcl", "wb") as f:
#     pickle.dump(fixed, f)

In [5]:
with open("pickles/fixed_pickle.pcl", "rb") as f:
    fixed:Fixed = pickle.load(f)

In [6]:
print(fixed.x_cg_max, fixed.x_cg_min, fixed.x_LE_wing, fixed.z_cg, fixed.z_tail_cone)
fixed.x_LE_wing = 1.255
delta_z = 0.01
fixed.z_tail_cone += delta_z
fixed.z_cg += delta_z
print(fixed.z_cg, fixed.z_tail_cone)
fixed.fuel_mass = 13.54
fixed.x_cg_min = 1.336 #m
fixed.x_cg_max = 1.403 #m
fixed.mass = 42.2 #kg 
fixed.fuselage.diameter_max = 0.33 # m

1.3589152301106575 1.271545179962353 1.4400000000000002 0.17957635338517247 0.11000000000000004
0.18957635338517248 0.12000000000000004


In [7]:
for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

# Creating full Aircraft objects

In [8]:
material_skin = Material(assumptions.cfrp_density, elastic_modulus=assumptions.cfrp_Young_modulus, 
                         poisson_ratio=assumptions.cfrp_poisson, shear_modulus=assumptions.cfrp_Young_modulus / 2 / (1 + assumptions.cfrp_poisson),
                         yield_strength=assumptions.cfrp_yield_strength, fracture_strength=assumptions.cfrp_yield_strength)

In [9]:
aircraft:list[Aircraft] = list()

for i, planform_recovered in enumerate(plaforms_recovered):
    main_wing = planform_recovered[0]
    planform_type = planform_recovered[1]

    ef = TailFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_h=max(4., main_wing.aspect_ratio/2)) if (planform_type == "tail") else CanardFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_c=max(5., main_wing.aspect_ratio/2))
    
    emp = ef.find_planforms(main_wing, print_=i==28)

    for e in emp:
        e.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
        e.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
        e.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
        e.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)
        e.mass_cache = 1
        e.x_cg_cache = .1

    aircraft_planforms = [main_wing] + emp #TODO add the empenage
    aircraft.append(Aircraft(
        fixed=fixed, #TODO: add the fuselage from CAD
        planforms=aircraft_planforms 
    ))

Stresses 17945691.48970201, 54812181.35113079, 0.0004
Stresses 12011958.946385456, 37879805.10591018, 0.0005959183673469389
Stresses 7931199.980387983, 30786416.695634246, 0.0004
Stresses 5245133.661045116, 20605002.445282508, 0.0005959183673469389
Stresses 3888254.7986966567, 15457728.041286655, 0.0007918367346938775
Stresses 13411867.278525403, 41926009.47740519, 0.0004
Stresses 5777444.856960322, 41284477.16928659, 0.0004
Stresses 3772791.5574911064, 27349206.84875104, 0.0005959183673469389
Stresses 2760131.643326246, 20291921.318375245, 0.0007918367346938775
Stresses 13422288.365042329, 41435027.62256522, 0.0004
Stresses 5782455.947697423, 40427473.555203676, 0.0004
Stresses 3776237.202064357, 26781687.354616795, 0.0005959183673469389
Stresses 2762786.495507451, 19871031.650789674, 0.0007918367346938775
Stresses 13422289.900154743, 41326815.55889486, 0.0004
Stresses 5782456.685844416, 40572600.04633272, 0.0004
Stresses 3776237.7096080026, 26877769.54070718, 0.0005959183673469389
St

In [10]:
s_ratios = [ac.planforms[1].wing_area / ac.planforms[0].wing_area for ac in aircraft]
print(s_ratios)
print(min(s_ratios), np.average(s_ratios), max(s_ratios))
ac_bad:Aircraft = aircraft[np.argmax(s_ratios)],
print(np.argmax(s_ratios))
ac_bad= ac_bad[0]
print(ac_bad.planforms[0].sweep_quarter_rad, ac_bad.planforms[0].aspect_ratio, ac_bad.planforms[0].cm_quarter_chord, ac_bad.planforms[0].thickness_to_chord)

[0.055816439670772655, 0.06195756147593051, 0.06614811786599539, 0.07083119734635873, 0.25319015413536056, 0.24963682397905815, 0.12891463110039753, 0.12016673200172062, 0.05556759972181553, 0.061684556978483435, 0.06640846381789658, 0.07112344293887264, 0.2529309816166052, 0.24935512832470164, 0.12862593823002869, 0.11988027706014165, 0.0551514660521477, 0.06122811074185525, 0.06684789007032714, 0.07161676351617219, 0.2524975817093381, 0.24888416761902113, 0.1281385752842055, 0.11939685666821351, 0.09598552071831921, 0.001569184533064235, 0.09772059785303312, 0.09970799526166406, 0.260888882233867, 0.26101064701296783, 0.17654098718833844, 0.17048202025895237, 0.09627940361072787, 0.0019640764924372797, 0.09627940361072787, 0.09770377055761559, 0.2605157940368704, 0.2605979498582125, 0.17842372008243634, 0.17244848585849398, 0.09498442249487775, 0.00022569917064009035, 0.09677545239064973, 0.09839294558299491, 0.26215985188264823, 0.2624172791434333, 0.17777570982900623, 0.17177133949

# Checking if reuirements are met

In [11]:
for ac in aircraft:
    ac.fixed = fixed

In [12]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq(),
    EmpennageReq(),
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear",
    "Empennage Requirement"
]

In [13]:
for ac in aircraft:
    failed_reqs = list()
    for requirement, label in zip(requirements, requirement_labels):
        # if type(requirement) == LGReq:
        #         print("f")
        if not requirement.assess(ac):
            failed_reqs.append(label)

    if len(failed_reqs):
        print(f"ac mass: {ac.total_mass()}, {ac.planforms[0].oswald}")
        print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
        print(f"Failed: {failed_reqs}")
        print()